In [ ]:
import pandas as pd 
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 10000)
pd.set_option('display.width', 1000)


sheets = pd.ExcelFile(f'./PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx')
print(sheets.sheet_names)

data = pd.read_excel(f'./PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx', sheet_name='Table1 (basic)', skiprows=[0])
data.drop(['QualityRequirement', 'QualNo', 'CustID','CustVer','InternalVer','MetalRavneQualityName','SteelGroup','Month','Year'], axis=1, inplace=True)
data.drop_duplicates(subset=['HeatNo'],keep='first',inplace=True)

data.dropna(subset=['Cr_Last_EOP','Mo_Last_EOP','Ni_Last_EOP','V_Last_EOP','W_Last_EOP'],inplace=True)


data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table2 (limits)',skiprows=[0])
data_.drop(['QualityRequirement','QualNo','CustID','CustVer','InternalVer','MetalRavneQualityName','SteelGroup','Month','Year'],axis=1,inplace=True)
data_.drop(['Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1,inplace=True)
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner') 

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table4 (alloys)',skiprows=[0])
data.drop(['LFVD_FeCrA', 'LFVD_FeCrC','LFVD_NiGran', 'LFVD_NiKatode','LFVD_FeMo', 'LFVD_Polymox','LFVD_FeV','LFVD_FeW72','LFVD_WPaketi'],axis=1,inplace=True)
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

# data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table7 (head)',skiprows=[0])
# data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

# data=data.merge(data_,on='HeatNo',how='inner') # толоько для нахождения брака у работников

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table8 (scrap)',skiprows=[0])
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table9 (all. scr.)',skiprows=[0])
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

data_=pd.read_excel('PQM - podatki za pilotni projekt (OCR12VM) - ENG.xlsx',sheet_name='Table10 (unall. scr.)',skiprows=[0])
data_.drop_duplicates(subset=['HeatNo'], keep='first', inplace=True)

data=data.merge(data_,on='HeatNo',how='inner')

display(data.head(3))
print('всего строк=',len(data))
print('всего столбцов=',len(data.columns))
data.to_csv('итоговая таблица.csv',sep=';',index=False)

### Удаление данных с нулевой диссперсией

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
data=pd.read_csv('итоговая таблица.csv',sep=';')
#LFVD_CaS LFVD_CoMet  LFVD_Cu LFVD_FeB	LFVD_FeBŽica	LFVD_FeCrNit  LFVD_FeMnA  LFVD_FeNbTa	LFVD_FeNbTafi13 LFVD_FeSiZrŽica	LFVD_FeTi	LFVD_FeTifi13
#LFVD_NiKatode	LFVD_NiOksid	LFVD_OdpCu  LFVD_SiMet  LFVD_KarboritZaVpih	LFVD_Ni90 

display(data.describe())
del_=[]
for i in data.describe().columns:
    if data.describe().loc['std',i]==0:
        del_.append(i)

del_.remove('Ni_LowerLimit')
del_.remove('Ni_Target')
del_.remove('W_LowerLimit')
del_.remove('W_Target')

data.drop(del_,axis=1,inplace=True)
data.drop(['Fill#1_ScrapName','Fill#1_ScrapWeight','Fill#2_ScrapName','Fill#2_ScrapWeight','Fill#3_ScrapName','Fill#3_ScrapWeight','Fill#4_ScrapName',
          'Fill#4_ScrapWeight','Fill#5_ScrapName','Fill#5_ScrapWeight','Fill#6_ScrapName','Fill#6_ScrapWeight'],axis=1,inplace=True)

print('всего строк=',len(data))
print('всего столбцов=',len(data.columns))
data.to_csv('итоговая таблица_испр.csv',sep=';',index=False)

### Проверка взаимных корреляций

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
data=pd.read_csv('итоговая таблица_испр.csv',sep=';')
data_=data.drop(['HeatNo','Date'],axis=1)

corr=data_.corr()

print('число столбцов в таблице=',len(data_.corr()))

display(corr.head(3))
gran_corr_value=0.8
corr_={i:[] for i in corr.columns}
for i in range(len(corr.columns)):    
    mas=corr[corr.columns[i]].tolist()
    el=mas[i]
    ind_max=[j for j in range(len(mas)) if (mas[j]>gran_corr_value)&(j!=i)]
    mas.remove(mas[i])
    corr_[corr.columns[i]]=[min(mas),max(mas)]+[corr.columns[j] for j in ind_max]
    print(corr.columns[i],':',corr_[corr.columns[i]])



### Гистограмма превышения целевого значения по компонентам

#### TODO
- создать гистограммы по отдельным сталеварам
- выявить случаи брака, превышение или занижение по элементу

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

data['Cr_exceed']=data.apply(lambda x:x['Cr_Final']-x['Cr_Target'],axis=1)
data['Ni_exceed']=data.apply(lambda x:x['Ni_Final']-x['Ni_Target'],axis=1)
data['Mo_exceed']=data.apply(lambda x:x['Mo_Final']-x['Mo_Target'],axis=1)
data['V_exceed']=data.apply(lambda x:x['V_Final']-x['V_Target'],axis=1)
data['W_exceed']=data.apply(lambda x:x['W_Final']-x['W_Target'],axis=1)

print('в среднем разница между Cr_Target и Cr_LowerLimit =',data.apply(lambda x:x['Cr_Target']-x['Cr_LowerLimit'],axis=1).mean())
print('в среднем разница между Ni_Target и Ni_LowerLimit =',data.apply(lambda x:x['Ni_Target']-x['Ni_LowerLimit'],axis=1).mean())
print('в среднем разница между Mo_Target и Mo_LowerLimit =',data.apply(lambda x:x['Mo_Target']-x['Mo_LowerLimit'],axis=1).mean())
print('в среднем разница между V_Target и V_LowerLimit =',data.apply(lambda x:x['V_Target']-x['V_LowerLimit'],axis=1).mean())
print('в среднем разница между W_Target и W_LowerLimit =',data.apply(lambda x:x['W_Target']-x['W_LowerLimit'],axis=1).mean())

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Histogram(x=data['Cr_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения Cr'))
fig.add_trace(go.Histogram(x=data['Ni_exceed'],xbins=dict(start=-0.5, end= 0.5, size=0.005), name='Гистограмма превышения Ni'))
fig.add_trace(go.Histogram(x=data['Mo_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения Mo'))
fig.add_trace(go.Histogram(x=data['V_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения V'))
fig.add_trace(go.Histogram(x=data['W_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения W'))
#fig.add_trace(go.Histogram(x=data['Cr_exceed'], name='Гистограмма превышения Cr'))
fig.show()

In [ ]:
data['Cr_adviser_target']=data.apply(lambda x:(x['Cr_Target']+x['Cr_LowerLimit'])/2,axis=1)
data['Ni_adviser_target']=data.apply(lambda x:(x['Ni_Target']+x['Ni_LowerLimit'])/2,axis=1)
data['Cr_exceed']=data.apply(lambda x:x['Cr_Final']-x['Cr_adviser_target'],axis=1)
data['Ni_exceed']=data.apply(lambda x:x['Ni_Final']-x['Ni_adviser_target'],axis=1)

print('в среднем разница между Cr_Target и Cr_LowerLimit =',data.apply(lambda x:x['Cr_Target']-x['Cr_LowerLimit'],axis=1).mean())
print('в среднем разница между Ni_Target и Ni_LowerLimit =',data.apply(lambda x:x['Ni_Target']-x['Ni_LowerLimit'],axis=1).mean())

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Histogram(x=data['Cr_exceed'],xbins=dict(start=-0.2, end= 0.2, size=0.005), name='Гистограмма превышения Cr'))
fig.add_trace(go.Histogram(x=data['Ni_exceed'],xbins=dict(start=-0.5, end= 0.5, size=0.005), name='Гистограмма превышения Ni'))
#fig.add_trace(go.Histogram(x=data['Cr_exceed'], name='Гистограмма превышения Cr'))
fig.show()

### Экономический эффект

In [ ]:
display(data[['TotalIngotsWeight','Cr_Last_EOP','Cr_Final','LFVD_FeCrA','LFVD_FeCrC',]].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_FeCrA']==0)&(data['LFVD_FeCrC']==0)&(data['LFVD_FeCrCSi']==0)&(data['LFVD_FeCrC51']==0)]))

display(data[['TotalIngotsWeight','Ni_Last_EOP','Ni_Final','LFVD_NiGran']].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_NiGran']==0)]))

display(data[['TotalIngotsWeight','Mo_Last_EOP','Mo_Final','LFVD_FeMo','LFVD_Polymox']].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_FeMo']==0)&(data['LFVD_Polymox']==0)]))

display(data[['TotalIngotsWeight','V_Last_EOP','V_Final','LFVD_FeV']].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_FeV']==0)]))

display(data[['TotalIngotsWeight','W_Last_EOP','W_Final','LFVD_FeW72']].head(3))
print('количество строк с большим исходным составом=',len(data[(data['LFVD_FeW72']==0)]))

In [ ]:
data_cr=data[(data['Cr_Final']>data['Cr_Last_EOP'])&(data['Cr_Final']>data['Cr_Target'])]
data_cr=data_cr.copy()

data_cr['FeCr_1%']=data_cr.apply(lambda x: (x['LFVD_FeCrA']+x['LFVD_FeCrC'])/(x['Cr_Final']-x['Cr_Last_EOP'])/x['TotalIngotsWeight']*100,axis=1)
data_cr['Cr_surplus']=data_cr.apply(lambda x:(x['LFVD_FeCrA']+x['LFVD_FeCrC'])-(x['Cr_Target']-x['Cr_Last_EOP'])*x['TotalIngotsWeight']*x['FeCr_1%']/100,axis=1)
display(data_cr[['TotalIngotsWeight','Cr_Last_EOP','Cr_Target','Cr_Final','LFVD_FeCrA','LFVD_FeCrC','FeCr_1%','Cr_surplus']].head(3))
print('перерасход в год FeCr=',round(data_cr['Cr_surplus'].sum()/10,0),'кг',' примерная стоимость=',round(data_cr['Cr_surplus'].sum()/10*300/1000000,2),'млн.руб')

In [ ]:
data_mo=data[(data['Mo_Final']>data['Mo_Last_EOP'])&(data['Mo_Final']>data['Mo_Target'])]
data_mo=data_mo.copy()

data_mo['FeMo_1%']=data_mo.apply(lambda x: (x['LFVD_FeMo']+x['LFVD_Polymox'])/(x['Mo_Final']-x['Mo_Last_EOP'])/x['TotalIngotsWeight']*100,axis=1)
data_mo['Mo_surplus']=data_mo.apply(lambda x:(x['LFVD_FeMo']+x['LFVD_Polymox'])-(x['Mo_Target']-x['Mo_Last_EOP'])*x['TotalIngotsWeight']*x['FeMo_1%']/100,axis=1)
display(data_mo[['TotalIngotsWeight','Mo_Last_EOP','Mo_Target','Mo_Final','LFVD_FeMo','FeMo_1%','Mo_surplus']].head(3))
print('перерасход в год Mo=',round(data_mo['Mo_surplus'].sum()/10,0),'кг',' примерная стоимость=',round(data_mo['Mo_surplus'].sum()/10*1500/1000000,2),'млн.руб')

In [ ]:
data_v=data[(data['V_Final']>data['V_Last_EOP'])&(data['V_Final']>data['V_Target'])]
data_v=data_v.copy()

data_v['FeV_1%']=data_v.apply(lambda x: (x['LFVD_FeV'])/(x['V_Final']-x['V_Last_EOP'])/x['TotalIngotsWeight']*100,axis=1)
data_v['V_surplus']=data_v.apply(lambda x:(x['LFVD_FeV'])-(x['V_Target']-x['V_Last_EOP'])*x['TotalIngotsWeight']*x['FeV_1%']/100,axis=1)
display(data_v[['TotalIngotsWeight','V_Last_EOP','V_Target','V_Final','LFVD_FeV','FeV_1%','V_surplus']].head(3))
print('перерасход в год V=',round(data_v['V_surplus'].sum()/10,0),'кг',' примерная стоимость=',round(data_v['V_surplus'].sum()/10*1500/1000000,2),'млн.руб')

In [ ]:
data_w=data[(data['W_Final']>data['W_Last_EOP'])&(data['W_Final']>data['W_Target'])]
data_w=data_w.copy()

data_w['FeW_1%']=data_w.apply(lambda x: (x['LFVD_FeW72'])/(x['W_Final']-x['W_Last_EOP'])/x['TotalIngotsWeight']*100,axis=1)
data_w['W_surplus']=data_w.apply(lambda x:(x['LFVD_FeW72'])-(x['W_Target']-x['W_Last_EOP'])*x['TotalIngotsWeight']*x['FeW_1%']/100,axis=1)
display(data_w[['TotalIngotsWeight','W_Last_EOP','W_Target','W_Final','LFVD_FeW72','FeW_1%','W_surplus']].head(3))
print('перерасход в год W=',round(data_w['W_surplus'].sum()/10,0),'кг',' примерная стоимость=',round(data_w['W_surplus'].sum()/10*1500/1000000,2),'млн.руб')

### Данные по отдельным сталеварам

In [ ]:
#'Cr_surplus', 'Mo_surplus', 'V_surplus', 'W_surplus', 'Total_defects' 
data_unique_emp = pd.DataFrame(data['LFVD_ShiftHead'].unique(), columns=['LFVD_ShiftHead'])

fig = make_subplots(specs=[[{"secondary_y": True}]])

for emp in data_unique_emp['LFVD_ShiftHead']:
    deffects = data.agg(lambda x: 1 if x['LFVD_ShiftHead']==emp and (x['Cr_Final'] > x['Cr_Last_EOP']) and (x['Cr_Final'] > x['Cr_Target']) else 0, axis=1).sum()
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Cr_defects']=deffects 
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']=deffects
    deffects = data.agg(lambda x: 1 if x['LFVD_ShiftHead']==emp and (x['Ni_Final'] > x['Ni_Last_EOP']) and (x['Ni_Final'] > x['Ni_Target']) else 0, axis=1).sum()
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Ni_defects']=deffects 
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']+=deffects
    deffects = data.agg(lambda x: 1 if x['LFVD_ShiftHead']==emp and (x['Mo_Final'] > x['Mo_Last_EOP']) and (x['Mo_Final'] > x['Mo_Target']) else 0, axis=1).sum()
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Mo_defects']=deffects 
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']+=deffects
    deffects = data.agg(lambda x: 1 if x['LFVD_ShiftHead']==emp and (x['V_Final'] > x['V_Last_EOP']) and (x['V_Final'] > x['V_Target']) else 0, axis=1).sum()
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'V_defects']=deffects 
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']+=deffects
    deffects = data.agg(lambda x: 1 if x['LFVD_ShiftHead']==emp and (x['W_Final'] > x['W_Last_EOP']) and (x['W_Final'] > x['W_Target']) else 0, axis=1).sum()
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'W_defects']=deffects
    data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']+=deffects

for emp in data_unique_emp:
    fig.add_trace(go.Bar(x=[emp], y=[data_unique_emp.loc[data_unique_emp['LFVD_ShiftHead']==emp, 'Total_defects']], name=f'Гистограмма дефектов для {emp}'))
    

fig.show()
display(data_unique_emp)



### Какая стоимость перерасхода если нам удастся обеспечить среднее значение между Cr_LowerLimit и Cr_Target?

### Выполните расчеты для каждого элемента: Mo, Ni, V, W

In [ ]:
import pandas as pd
import dtreeviz
from sklearn.tree import DecisionTreeClassifier


random_state=42
data=pd.read_csv('итоговая таблица_испр.csv',sep=';')
data.fillna(0,inplace=True)
#display(data)
data['quality']=data.apply(lambda x: 1 if x['Cr_Final']<=x['Cr_Target'] else 0,axis=1)

X=data.drop(['HeatNo','quality','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).values
y=data['quality'].values

features=data.drop(['HeatNo','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).columns.tolist()
class_names=list(set(y))
target='quality'
tree_classifier = DecisionTreeClassifier(max_depth=33, random_state=random_state)
tree_classifier.fit(X, y)

# viz_model = dtreeviz.model(tree_classifier,
#                            X_train=X, y_train=y,
#                            feature_names=features,
#                            target_name=target)

#viz_model.view(scale=0.8)
#viz_model.save('dtreeviz.svg')

array([[49440.,   600.,  2000., ..., 40000.,     0.,     0.],
       [46800.,  1000.,   100., ..., 40200.,     0.,     0.],
       [45320.,   800.,  2000., ..., 40200.,     0.,     0.],
       ...,
       [44400.,   600.,   300., ...,     0., 22000.,     0.],
       [48410.,   800.,   700., ...,     0., 24000.,     0.],
       [51820.,   800.,   300., ...,     0., 25100.,     0.]],
      shape=(3225, 74))

In [14]:
import pandas as pd
import dtreeviz
from sklearn.tree import DecisionTreeRegressor
import dtreeviz

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

random_state=42
data=pd.read_csv('итоговая таблица_испр.csv',sep=';')
data.fillna(0,inplace=True)
trees = []
#display(data)
qual = ['Cr_Final', 'Ni_Final','Mo_Final','V_Final', 'W_Final']
for i in range(5):
    X=data.drop(['HeatNo','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).values #['HeatNo','quality','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final']
    y=data[qual[i]].values
    features=data.drop(['HeatNo','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).columns.tolist()  #['HeatNo','quality','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final']
    tree_reggression = DecisionTreeRegressor(max_depth=33, random_state=random_state)
    trees.append(tree_reggression.fit(X, y))
trees = dict(zip(qual, trees))
print(trees)
print('Деревья обучены')

{'Cr_Final': DecisionTreeRegressor(max_depth=33, random_state=42), 'Ni_Final': DecisionTreeRegressor(max_depth=33, random_state=42), 'Mo_Final': DecisionTreeRegressor(max_depth=33, random_state=42), 'V_Final': DecisionTreeRegressor(max_depth=33, random_state=42), 'W_Final': DecisionTreeRegressor(max_depth=33, random_state=42)}
Деревья обучены


### Метрики точности моделей
Здесь используем оценки:
1. mE = (1/n)sum(y`i - yi)
2. MAE. MSE, RMSE
3. R correlation

In [15]:
#/////////////// UNDER CONSTRUCTION BLOCK ///////////////////
data=pd.read_csv('итоговая таблица_испр.csv',sep=';')
data.fillna(0,inplace=True)
check = data.drop(['HeatNo','Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1).T.values
ans = pd.DataFrame()
for i in check:
    for j in trees:
        ans[j] = trees[j].predict(i)
display(ans)


ValueError: Expected 2D array, got 1D array instead:
array=[49440. 46800. 45320. ... 44400. 48410. 51820.].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

Не понимаю как отфильтровать значения ближайшие к параметрам

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

import random as rnd
example_data = data.iloc[rnd.randint(0, len(data))].loc[['PouringScrap', 'OtherScrap','TotalIngotsWeight', 'Cr_Last_EOP', 
                                                         'Cr_LowerLimit', 'Cr_UpperLimit', 'Cr_Target', 
                                                         'Ni_Last_EOP', 'Ni_LowerLimit', 'Ni_UpperLimit', 'Ni_Target',
                                                         'Mo_Last_EOP', 'Mo_LowerLimit', 'Mo_UpperLimit', 'Mo_Target',
                                                         'V_Last_EOP', 'V_LowerLimit', 'V_UpperLimit', 'V_Target',
                                                         'W_Last_EOP', 'W_LowerLimit', 'W_UpperLimit', 'W_Target']]
other_par = data.drop(data.columns[:31], axis=1)
ans = pd.DataFrame()

for i in range(0, len(other_par)):
    data_ = pd.DataFrame(pd.concat([example_data, other_par.iloc[i]], axis=0)).T
    
    if trees[0][1].predict(data_.values) == 1:
        ans = pd.concat([ans, data_], ignore_index=True)
display(ans)


In [16]:
import pandas as pd
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

import random as rnd

data = pd.read_csv('итоговая таблица_испр.csv', sep=';')

example_data = data.iloc[rnd.randint(0, len(data))].loc[['PouringScrap', 'OtherScrap','TotalIngotsWeight', 'Cr_Last_EOP', 
                                                         'Cr_LowerLimit', 'Cr_UpperLimit', 'Cr_Target']].to_frame() #data from historical
example_data = pd.DataFrame(data={'PouringScrap': [600], 'OtherScrap': [200], 'TotalIngotsWeight': [33300],
                                   'Cr_Last_EOP':[10.2019], 'Cr_LowerLimit':[11.1], 'Cr_UpperLimit':[11.75], 'Cr_Target':[11.46]}).T #random data
other_par = data.drop(['HeatNo', 'Date','Cr_Final','Ni_Final','Mo_Final','V_Final','W_Final'],axis=1)
ans = pd.DataFrame(columns=other_par.columns.to_list())
#display(ans)

for i in range(0, len(other_par)):
    miss = other_par.columns.difference(example_data.T.columns).to_list()
    data_ = example_data.T
    # display(len(miss))
    # display(data_)
    # display(other_par.loc[i, miss].to_frame().T)
    
    other_par.loc[i, miss].to_frame().T
    data_[miss] = other_par.loc[i, miss].to_list()
    # display(len(data_.columns))
    if trees[0][1].predict(data_.values) == 1:
        ans = pd.concat([ans, data_], ignore_index=True)
display(ans)


KeyError: 0

In [ ]:
import pandas as pd


